In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset, random_split
from torch.nn.utils.rnn import pad_sequence
import time
import copy
import os
from pathlib import Path
import glob
import pandas as pd
import numpy as np
import timeit
import time

In [13]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"
device = get_device()
device

'cpu'

In [19]:
class SleepDataset(Dataset):
    def __init__(self, file_path, length, window_size=5):
        
        self.file_path = file_path
        self.length = length
        self.data = torch.load(self.file_path, weights_only=False, mmap=True)
#         self.features = torch.from_numpy(np.transpose(data["X"], axes=(0, 2, 1))).float()
#         self.labels = torch.from_numpy(data["y"]).long()


    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        
        # return self.features[idx], self.labels[idx]
        feature = torch.from_numpy(self.data["X"][idx]).float()
        label = torch.tensor(self.data["y"][idx]).long()
        
        # Apply transpose here if needed based on original shape
        feature = feature.transpose(1, 0)
        return feature, label
        

data_dir = 'anphy_sleep_data/patient_records/clean'       
file_paths = glob.glob(os.path.join(data_dir, '*.pt'))



In [4]:
# import csv
# lengths = [torch.load(file_path, weights_only=False)["y"].shape[0] for file_path in file_paths]
# with open('recording_epoch_nums.csv', 'w', newline='') as f:
#     writer = csv.writer(f)
#     for i in range(len(file_paths)):
#         writer.writerow([file_paths[i].replace(f"{data_dir}/", ""), lengths[i]]) # writerow takes a list

In [5]:
test_load = np.load(data_dir + "/EPCTL01_dataset.pt", mmap_mode='r', allow_pickle=True)
test_load

NpzFile 'anphy_sleep_data/patient_records/clean/EPCTL01_dataset.pt' with keys: EPCTL01_dataset/data.pkl, EPCTL01_dataset/.format_version, EPCTL01_dataset/.storage_alignment, EPCTL01_dataset/byteorder, EPCTL01_dataset/version...

In [ ]:
test_load["EPCTL01_/.format_version"]

In [11]:
pickle_test = test_load["EPCTL01_dataset/data.pkl"]
pickle_test

In [15]:
metadata = pd.read_csv("recording_epoch_nums.csv", header=None)
metadata.sort_values(by=0, inplace=True)
metadata = metadata.reset_index(drop=True)
metadata

,0,1
0,EPCTL01_dataset.pt,957
1,EPCTL02_dataset.pt,1072
2,EPCTL03_dataset.pt,843
3,EPCTL04_dataset.pt,762
4,EPCTL05_dataset.pt,755
5,EPCTL06_dataset.pt,919
6,EPCTL07_dataset.pt,979
7,EPCTL08_dataset.pt,873
8,EPCTL09_dataset.pt,945
9,EPCTL10_dataset.pt,950


In [ ]:
metadata.loc[:, 1].sum()

In [20]:

datasets = [SleepDataset(data_dir + "/" + metadata.loc[i, 0], metadata.loc[i, 1]) for i in range(len(metadata))]

In [21]:
combined_dataset = ConcatDataset(datasets)

In [22]:
combined_dataset[0][0].shape

torch.Size([3000, 14])

In [23]:
len(combined_dataset)

26118

In [10]:
def custom_collate(batch):
#     print(f"Length of batch: {len(batch)}")
#     print(f"First item in batch's first element: {batch[0][0].shape}")
#     print(f"First item in batch's second element: {batch[0][1].shape}")
    data = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    # Pad sequences to the length of the longest in the batch
#     print(f"Data list: {data}")
#     print(f"Labels list: {labels}" )
    padded_data = pad_sequence(data, batch_first=True)
    labels = torch.tensor(labels)
    return padded_data, labels


In [11]:
class MyModel(nn.Module):
    # You can use pre-existing models but change layers to recieve full credit.
    def __init__(self):
        super(MyModel, self).__init__()
        ### TODO: BEGIN SOLUTION ###
        self.model_sequential = nn.Sequential(
            nn.Conv1d(in_channels=14, out_channels= 32, kernel_size=3),
            nn.BatchNorm1d(num_features=32),
            nn.ReLU(),
            nn.MaxPool1d(2, 2),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.2),

            nn.Conv1d(in_channels=64, out_channels = 128, kernel_size=3),
            nn.BatchNorm1d(num_features=128),
            nn.ReLU(),
            nn.Conv1d(in_channels=128, out_channels = 256, kernel_size=3),
            nn.BatchNorm1d(num_features=256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.2),
            
            nn.Conv1d(in_channels=256, out_channels = 64, kernel_size=7),
            nn.BatchNorm1d(num_features=64),
            nn.ReLU(),
            nn.Conv1d(in_channels=64, out_channels = 32, kernel_size=3),
            nn.BatchNorm1d(num_features=32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.2),

            nn.Flatten(),
            
            nn.Linear(5824, 1028),
            nn.ReLU(),
            nn.Linear(1028, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 6),
        )
        ### END SOLUTION ###

    def forward(self, x):
        outs = None
        ### TODO: BEGIN SOLUTION ###
        outs = self.model_sequential(x)
        ### END SOLUTION ###
        return outs

In [14]:


class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def accuracy(output, target):
    """Computes the precision@k for the specified values of k"""
    batch_size = target.shape[0]

    _, pred = torch.max(output, dim=-1)

    correct = pred.eq(target).sum() * 1.0

    acc = correct.item() / batch_size

    return acc


def train(epoch, data_loader, model, optimizer, criterion):
    iter_time = AverageMeter()
    losses = AverageMeter()
    acc = AverageMeter()

    for idx, (data, target) in enumerate(data_loader):
        start = time.time()

        data = data.to(device)
        data = torch.transpose(data, 1, 2)
        # print(f"data shape: {data.shape}")
        target = target.to(device)

        # calculate model predictions, training loss, and update model parameters
        ### TODO: BEGIN SOLUTION ###
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        
        ### END SOLUTION ###

        batch_acc = accuracy(out, target)

        losses.update(loss, out.shape[0])
        acc.update(batch_acc, out.shape[0])

        iter_time.update(time.time() - start)
        if idx % len(data_loader) == 0:
            print(
                (
                    "Epoch: [{0}][{1}/{2}]\t"
                    "Time {iter_time.val:.3f} ({iter_time.avg:.3f})\t"
                    "Loss {loss.val:.4f} ({loss.avg:.4f})\t"
                    "Prec @1 {top1.val:.4f} ({top1.avg:.4f})\t"
                ).format(
                    epoch,
                    idx,
                    len(data_loader),
                    iter_time=iter_time,
                    loss=losses,
                    top1=acc,
                )
            )


def validate(epoch, val_loader, model, criterion):
    """
    Hint: make sure to use torch.no_grad() to disable gradient computation. This will help reduce memory usage and speed up computation.
    """
    iter_time = AverageMeter()
    losses = AverageMeter()
    acc = AverageMeter()

    num_class = 6
    cm = torch.zeros(num_class, num_class)
    # evaluation loop
    for idx, (data, target) in enumerate(val_loader):
        start = time.time()

        data = data.to(device)
        data = torch.transpose(data, 1, 2)
        target = target.to(device)

        # calculate model predictions and validation loss
        ### TODO: BEGIN SOLUTION ###
        with torch.no_grad():
            out = model(data)
            loss = criterion(out, target)

            
        ### END SOLUTION ###

        batch_acc = accuracy(out, target)

        # update confusion matrix
        _, preds = torch.max(out, 1)
        for t, p in zip(target.view(-1), preds.view(-1)):
            cm[t.long(), p.long()] += 1

        losses.update(loss, out.shape[0])
        acc.update(batch_acc, out.shape[0])

        iter_time.update(time.time() - start)
        if idx % len(val_loader) == 0:
            print(
                (
                    "Epoch: [{0}][{1}/{2}]\t"
                    "Time {iter_time.val:.3f} ({iter_time.avg:.3f})\t"
                ).format(
                    epoch,
                    idx,
                    len(val_loader),
                    iter_time=iter_time,
                    loss=losses,
                    top1=acc,
                )
            )
    cm = cm / cm.sum(1)
    per_cls_acc = cm.diag().detach().numpy().tolist()
    for i, acc_i in enumerate(per_cls_acc):
        print("Accuracy of Class {}: {:.4f}".format(i, acc_i))

    print("* Prec @1: {top1.avg:.4f}".format(top1=acc))
    return acc.avg, cm


def adjust_learning_rate(optimizer, learning_rate, epoch, warmup, steps):
    epoch += 1
    if epoch <= warmup:
        lr = learning_rate * epoch / warmup
    elif epoch > steps[1]:
        lr = learning_rate * 0.01
    elif epoch > steps[0]:
        lr = learning_rate * 0.1
    else:
        lr = learning_rate
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr


In [15]:
def main(lr, momentum, weight_decay, epochs, warmup, steps):
    device = get_device()
    
    model = MyModel()
    model = model.to(device)
    
    train_size = int(len(combined_dataset) * 0.8)
    test_size = len(combined_dataset) - train_size
    train_dataset, test_dataset = random_split(combined_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True, collate_fn=custom_collate)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True, collate_fn=custom_collate)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr,
        momentum,
        weight_decay
    )
        
    best = 0.0
    best_cm = None
    best_model = model
    for epoch in range(epochs):
        adjust_learning_rate(optimizer, lr, epoch, warmup, steps)

        # train loop
        train(epoch, train_loader, model, optimizer, criterion)

        # validation loop
        acc, cm = validate(epoch, test_loader, model, criterion)

        if acc > best:
            best = acc
            best_cm = cm
            best_model = copy.deepcopy(model)

    print("Best Prec @1 Acccuracy: {:.4f}".format(best))
    per_cls_acc = best_cm.diag().detach().numpy().tolist()
    for i, acc_i in enumerate(per_cls_acc):
        print("Accuracy of Class {}: {:.4f}".format(i, acc_i))
    
    if not os.path.exists("./checkpoints"):
        os.makedirs("./checkpoints")
        torch.save(
            best_model.state_dict(), "./checkpoints/" + "best_model" + ".pth"
        )


In [ ]:
main(lr = 0.01, momentum = 0.9, weight_decay = 0.001, epochs = 1000, warmup = 0, steps = [6, 8])

/home/hice1/eyang82/.conda/envs/anphy_sleep_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch: [0][0/653]	Time 4.934 (4.934)	Loss 1.7632 (1.7632)	Prec @1 0.1875 (0.1875)	
Epoch: [0][0/164]	Time 0.242 (0.242)	
Accuracy of Class 0: 0.3785
Accuracy of Class 1: 0.4310
Accuracy of Class 2: 0.0000
Accuracy of Class 3: 0.8547
Accuracy of Class 4: 0.5224
Accuracy of Class 5: 0.4365
* Prec @1: 0.5655
Epoch: [1][0/653]	Time 0.239 (0.239)	Loss 1.0238 (1.0238)	Prec @1 0.5938 (0.5938)	


In [ ]:
m